# 99 · Manual source-control and release review

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Read-only by default. Never commit raw images, per-person labels, private predictions, source records, tokens or checkpoints automatically.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Inspect the checkout

In [ ]:
import subprocess
if (REPO/'.git').exists():
    print(subprocess.check_output(['git','-C',str(REPO),'status','--short'],text=True))
else:
    print('This archive is not connected to your GitHub repository. No live repository has been modified.')

## 2. Clear notebook output metadata only in an explicitly selected publication copy

In [ ]:
import nbformat
CLEAR_NOTEBOOK_OUTPUTS=False
if CLEAR_NOTEBOOK_OUTPUTS:
    for path in (REPO/'notebooks').glob('*.ipynb'):
        nb=nbformat.read(path,as_version=4)
        for cell in nb.cells:
            if cell.cell_type=='code':cell.outputs=[];cell.execution_count=None
        nbformat.write(nb,path)
print('No private research data have been staged.')

## 3. Review staged files before any manual commit

In [ ]:
if (REPO/'.git').exists():
    subprocess.run([sys.executable,str(REPO/'scripts/git_review.py')],cwd=REPO,check=True)
print('Commit and push only reviewed code and permitted aggregate outputs from your authenticated Git client.')
print('Use a development branch. Never force-push over the existing project or include credentials in notebook cells.')

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
